# 03 - Entrenar Modelo 2 (multi-label, 5 patógenos) 

Una hoja puede tener varias enfermedades simultáneamente. Sigmoid + Asymmetric Loss (SOTA multi-label).

Clases: `bacterianas`, `fungicas`, `plagas_insectos`, `roya`, `virales`.

Pipeline mejorado:
1. EfficientNetB0 backbone + cabeza sigmoid(N)
2. **Asymmetric Loss** (Ridnik 2021) — supera BinaryFocalCrossentropy en multi-label
3. **CutMix + MixUp** — co-ocurrencia explícita con etiquetas multi-hot continuas
4. **Augmentación de color reforzada** (ChannelShuffle + RGBShift + HSV ampliado) para separar bacterianas/fungicas
5. **EMA tracking** durante fine-tune
6. Label smoothing **dentro de la loss** (no en generator) → arregla AUC monitor
7. Calibración de threshold por clase maximizando F1 en validación
8. Salida: `model2_pathogen.keras` + `thresholds.json`

In [ ]:
!pip install -q tensorflow albumentations scikit-learn matplotlib

In [ ]:
import json
from pathlib import Path
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

tf.random.set_seed(42)
np.random.seed(42)
print("GPU:", tf.config.list_physical_devices("GPU"))

IMG_SIZE = (224, 224)
BATCH = 16
EPOCHS_P1 = 20
EPOCHS_P2 = 45
LR_P1 = 1e-3
LR_P2 = 1e-5
EMA_DECAY = 0.9999
MIX_PROB_CUTMIX = 0.20
MIX_PROB_MIXUP = 0.10
LABEL_SMOOTH = 0.05
ASL_GAMMA_NEG = 5.0
ASL_GAMMA_POS = 0.0
ASL_CLIP = 0.05

DATA = Path("./splits/clasificacion_patogeno")
OUT = Path("./outputs")
OUT.mkdir(exist_ok=True)


In [ ]:
import time
import json
from pathlib import Path
import numpy as np
import albumentations as A
from PIL import Image
from tensorflow.keras.utils import Sequence
from tensorflow.keras.applications.efficientnet import preprocess_input

DATA = globals().get("DATA", Path("./splits/clasificacion_patogeno"))
OUT = globals().get("OUT", Path("./outputs"))
OUT.mkdir(exist_ok=True)
IMG_SIZE = globals().get("IMG_SIZE", (224, 224))
BATCH = globals().get("BATCH", 16)
MIX_PROB_CUTMIX = globals().get("MIX_PROB_CUTMIX", 0.2)
MIX_PROB_MIXUP = globals().get("MIX_PROB_MIXUP", 0.1)

TRAIN_AUG = A.Compose([
    A.Rotate(limit=45, border_mode=0, p=0.7),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.3),
    A.RandomBrightnessContrast(brightness_limit=0.25, contrast_limit=0.25, p=0.6),
    A.HueSaturationValue(hue_shift_limit=20, sat_shift_limit=30, val_shift_limit=20, p=0.5),
    A.ChannelShuffle(p=0.1),
    A.RGBShift(r_shift_limit=15, g_shift_limit=15, b_shift_limit=15, p=0.25),
    A.OneOf([
        A.GaussianBlur(blur_limit=(3, 5), p=1.0),
        A.MotionBlur(blur_limit=5, p=1.0),
    ], p=0.2),
    A.GaussNoise(std_range=(0.02, 0.1), p=0.15),
    A.Affine(translate_percent=0.08, scale=(0.88, 1.15), rotate=0, p=0.4),
    A.CoarseDropout(
        num_holes_range=(1, 4),
        hole_height_range=(16, 28),
        hole_width_range=(16, 28),
        fill=0,
        p=0.1,
    ),
])
VAL_AUG = A.Compose([])


def safe_read_image(fp, target_size, retries=3, delay=0.5):
    for attempt in range(retries):
        try:
            return np.array(Image.open(fp).convert("RGB").resize(target_size))
        except (OSError, IOError) as exc:
            if attempt < retries - 1:
                time.sleep(delay)
                continue
            print(f"[warn] saltando {fp}: {exc}")
            return None


def multi_hot(label_idx: int, num_classes: int) -> np.ndarray:
    vector = np.zeros(num_classes, dtype=np.float32)
    vector[label_idx] = 1.0
    return vector


def cutmix_pair(img_a, y_a, img_b, y_b):
    lam = np.random.beta(1.0, 1.0)
    h, w = img_a.shape[:2]
    cut_w = int(w * np.sqrt(1 - lam))
    cut_h = int(h * np.sqrt(1 - lam))
    cx = np.random.randint(w)
    cy = np.random.randint(h)
    x1 = max(cx - cut_w // 2, 0)
    y1 = max(cy - cut_h // 2, 0)
    x2 = min(cx + cut_w // 2, w)
    y2 = min(cy + cut_h // 2, h)
    mixed = img_a.copy()
    mixed[y1:y2, x1:x2] = img_b[y1:y2, x1:x2]
    real_lam = 1 - ((x2 - x1) * (y2 - y1) / (w * h))
    return mixed, y_a * real_lam + y_b * (1 - real_lam)


def mixup_pair(img_a, y_a, img_b, y_b, alpha=0.4):
    lam = np.random.beta(alpha, alpha)
    return (lam * img_a + (1 - lam) * img_b).astype(np.float32), lam * y_a + (1 - lam) * y_b


class MultiLabelLeafSequence(Sequence):
    def __init__(self, directory, img_size, batch_size, augment, shuffle,
                 cutmix=False, mixup=False):
        self.img_size = img_size
        self.batch_size = batch_size
        self.augment = augment
        self.shuffle = shuffle
        self.cutmix = cutmix
        self.mixup = mixup
        self.class_indices = {}
        self.samples = []
        classes = sorted(p.name for p in Path(directory).iterdir() if p.is_dir())
        for idx, name in enumerate(classes):
            self.class_indices[name] = idx
            for ext in ("*.jpg", "*.jpeg", "*.png", "*.bmp"):
                for fp in (Path(directory) / name).glob(ext):
                    self.samples.append((str(fp), idx))
        self.n = len(self.samples)
        self.num_classes = len(classes)
        if shuffle:
            self._shuffle()

    def _shuffle(self):
        np.random.shuffle(self.samples)

    def __len__(self):
        return max(1, self.n // self.batch_size)

    def __getitem__(self, batch_idx):
        batch = self.samples[batch_idx * self.batch_size:(batch_idx + 1) * self.batch_size]
        images, labels = [], []
        for fp, cls in batch:
            img = safe_read_image(fp, self.img_size)
            if img is None:
                continue
            if self.augment:
                img = TRAIN_AUG(image=img)["image"]
            else:
                img = VAL_AUG(image=img)["image"]
            images.append(img)
            labels.append(multi_hot(cls, self.num_classes))
        if not images:
            placeholder = np.zeros((1, *self.img_size, 3), dtype=np.float32)
            placeholder_label = np.zeros((1, self.num_classes), dtype=np.float32)
            return preprocess_input(placeholder), placeholder_label
        images = np.stack(images).astype(np.float32)
        labels = np.stack(labels)
        if self.augment:
            rng = np.random.rand()
            if self.cutmix and rng < MIX_PROB_CUTMIX:
                images, labels = self._mix_batch(images, labels, cutmix_pair)
            elif self.mixup and rng < MIX_PROB_CUTMIX + MIX_PROB_MIXUP:
                images, labels = self._mix_batch(images, labels, mixup_pair)
        return preprocess_input(images), labels

    def _mix_batch(self, images, labels, mix_fn):
        perm = np.random.permutation(len(images))
        out_imgs = np.empty_like(images)
        out_labels = np.empty_like(labels)
        for i in range(len(images)):
            out_imgs[i], out_labels[i] = mix_fn(
                images[i], labels[i], images[perm[i]], labels[perm[i]]
            )
        return out_imgs, out_labels

    def on_epoch_end(self):
        if self.shuffle:
            self._shuffle()


train_gen = MultiLabelLeafSequence(
    DATA / "train", IMG_SIZE, BATCH, augment=True, shuffle=True,
    cutmix=True, mixup=True,
)
val_gen = MultiLabelLeafSequence(
    DATA / "val", IMG_SIZE, BATCH, augment=False, shuffle=False,
)
NUM_CLASSES = train_gen.num_classes
CLASS_NAMES = [k for k, _ in sorted(train_gen.class_indices.items(), key=lambda kv: kv[1])]
print(f"Train: {train_gen.n} | Val: {val_gen.n} | Clases: {CLASS_NAMES}")
with open(OUT / "class_indices_model2_pathogen.json", "w") as f:
    json.dump(train_gen.class_indices, f)

In [ ]:
import tensorflow as tf
import numpy as np

IMG_SIZE = globals().get("IMG_SIZE", (224, 224))
LR_P1 = globals().get("LR_P1", 1e-3)
LABEL_SMOOTH = globals().get("LABEL_SMOOTH", 0.05)
ASL_GAMMA_NEG = globals().get("ASL_GAMMA_NEG", 4.0)
ASL_GAMMA_POS = globals().get("ASL_GAMMA_POS", 0.0)
ASL_CLIP = globals().get("ASL_CLIP", 0.05)
EMA_DECAY = globals().get("EMA_DECAY", 0.9999)
NUM_CLASSES = globals().get("NUM_CLASSES", 5)
CLASS_NAMES = globals().get("CLASS_NAMES", [])


class AsymmetricLoss(tf.keras.losses.Loss):
    def __init__(self, gamma_neg=4.0, gamma_pos=0.0, clip=0.05, label_smoothing=0.05,
                 class_weights=None, name="asymmetric_loss", **kwargs):
        super().__init__(name=name, **kwargs)
        self.gamma_neg = gamma_neg
        self.gamma_pos = gamma_pos
        self.clip = clip
        self.label_smoothing = label_smoothing
        self.class_weights = class_weights
        self.eps = 1e-8

    def call(self, y_true, y_pred):
        y_true = tf.cast(y_true, tf.float32)
        y_pred = tf.cast(y_pred, tf.float32)
        num_classes = tf.cast(tf.shape(y_true)[-1], tf.float32)
        y_true_s = y_true * (1.0 - self.label_smoothing) + self.label_smoothing / num_classes
        xs_pos = y_pred
        xs_neg = 1.0 - y_pred
        if self.clip > 0:
            xs_neg = tf.clip_by_value(xs_neg + self.clip, 0.0, 1.0)
        log_pos = tf.math.log(tf.clip_by_value(xs_pos, self.eps, 1.0))
        log_neg = tf.math.log(tf.clip_by_value(xs_neg, self.eps, 1.0))
        loss_pos = y_true_s * log_pos
        loss_neg = (1.0 - y_true_s) * log_neg
        if self.gamma_pos > 0:
            loss_pos = loss_pos * tf.pow(1.0 - xs_pos, self.gamma_pos)
        if self.gamma_neg > 0:
            loss_neg = loss_neg * tf.pow(1.0 - xs_neg, self.gamma_neg)
        per_class = loss_pos + loss_neg
        if self.class_weights is not None:
            per_class = per_class * tf.constant(self.class_weights, tf.float32)
        return -tf.reduce_sum(per_class, axis=-1)

    def get_config(self):
        base = super().get_config()
        base.update({
            "gamma_neg": self.gamma_neg,
            "gamma_pos": self.gamma_pos,
            "clip": self.clip,
            "label_smoothing": self.label_smoothing,
            "class_weights": self.class_weights,
        })
        return base


class EMACallback(tf.keras.callbacks.Callback):
    def __init__(self, decay=0.9999):
        super().__init__()
        self.decay = decay
        self.ema_weights = None

    def on_train_begin(self, logs=None):
        self.ema_weights = [w.numpy().copy() for w in self.model.trainable_weights]

    def on_train_batch_end(self, batch, logs=None):
        for i, w in enumerate(self.model.trainable_weights):
            self.ema_weights[i] = self.decay * self.ema_weights[i] + (1 - self.decay) * w.numpy()

    def on_train_end(self, logs=None):
        for w, ew in zip(self.model.trainable_weights, self.ema_weights):
            w.assign(ew)


def build_multilabel_model(num_classes: int) -> tf.keras.Model:
    base = tf.keras.applications.EfficientNetB0(
        weights="imagenet", include_top=False, input_shape=(*IMG_SIZE, 3)
    )
    base.trainable = False
    x = base.output
    x = tf.keras.layers.GlobalAveragePooling2D()(x)
    x = tf.keras.layers.BatchNormalization()(x)
    x = tf.keras.layers.Dropout(0.3)(x)
    x = tf.keras.layers.Dense(
        256, activation="relu", kernel_regularizer=tf.keras.regularizers.l2(5e-5)
    )(x)
    x = tf.keras.layers.Dropout(0.2)(x)
    output = tf.keras.layers.Dense(num_classes, activation="sigmoid")(x)
    return tf.keras.Model(base.input, output, name="model2_pathogen_multilabel")


_counts = np.zeros(NUM_CLASSES)
for _, _idx in train_gen.samples:
    _counts[_idx] += 1
_w = _counts.sum() / (NUM_CLASSES * np.clip(_counts, 1, None))
CLASS_WEIGHTS = (_w / _w.mean()).tolist()
print('class_weights M2:', dict(zip(CLASS_NAMES, [round(w, 3) for w in CLASS_WEIGHTS])))


ASYMMETRIC_LOSS = AsymmetricLoss(
    gamma_neg=ASL_GAMMA_NEG,
    gamma_pos=ASL_GAMMA_POS,
    clip=ASL_CLIP,
    label_smoothing=LABEL_SMOOTH,
    class_weights=CLASS_WEIGHTS,
)
METRICS = [
    tf.keras.metrics.BinaryAccuracy(name="bin_acc"),
    tf.keras.metrics.AUC(multi_label=True, name="auc"),
    tf.keras.metrics.Precision(name="prec"),
    tf.keras.metrics.Recall(name="rec"),
]

model = build_multilabel_model(NUM_CLASSES)
model.compile(optimizer=tf.keras.optimizers.Adam(LR_P1), loss=ASYMMETRIC_LOSS, metrics=METRICS)
model.summary()

In [ ]:
from pathlib import Path
import tensorflow as tf

EPOCHS_P1 = globals().get("EPOCHS_P1", 20)
OUT = globals().get("OUT", Path("./outputs"))

callbacks_p1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_auc", mode="max", patience=6,
        restore_best_weights=True, verbose=1,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(OUT / "model2_pathogen_p1_best.keras"),
        monitor="val_auc", mode="max", save_best_only=True, verbose=1,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1,
    ),
]

print("=== FASE 1: cabeza congelada ===")
h1 = model.fit(
    train_gen, validation_data=val_gen,
    epochs=EPOCHS_P1, callbacks=callbacks_p1, verbose=1,
)


In [ ]:
from pathlib import Path
import numpy as np
import tensorflow as tf

EPOCHS_P2 = globals().get("EPOCHS_P2", 45)
LR_P2 = globals().get("LR_P2", 1e-5)
EMA_DECAY = globals().get("EMA_DECAY", 0.9999)
BATCH = globals().get("BATCH", 16)
OUT = globals().get("OUT", Path("./outputs"))

_h1 = globals().get("h1", None)
_initial_epoch = len(_h1.history["loss"]) if _h1 is not None else 0

for layer in model.layers:
    layer.trainable = True
for layer in model.layers[:-80]:
    layer.trainable = False
for layer in model.layers:
    if isinstance(layer, tf.keras.layers.BatchNormalization):
        layer.trainable = False

trainable_count = sum(1 for l in model.layers if l.trainable)
print(f"Layers entrenables en phase 2: {trainable_count}")

_steps = max(1, train_gen.n // BATCH) * EPOCHS_P2
_lr_schedule = tf.keras.optimizers.schedules.CosineDecay(
    initial_learning_rate=LR_P2,
    decay_steps=_steps,
    alpha=0.01,
)

model.compile(
    optimizer=tf.keras.optimizers.Adam(_lr_schedule),
    loss=ASYMMETRIC_LOSS,
    metrics=METRICS,
)

callbacks_p2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss", patience=12,
        restore_best_weights=True, verbose=1,
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(OUT / "model2_pathogen_p2_best.keras"),
        monitor="val_auc", mode="max", save_best_only=True, verbose=1,
    ),
    EMACallback(decay=EMA_DECAY),
]

print("=== FASE 2: fine-tuning ultimos 80 layers + Cosine LR + EMA ===")
h2 = model.fit(
    train_gen, validation_data=val_gen,
    epochs=_initial_epoch + EPOCHS_P2, initial_epoch=_initial_epoch,
    callbacks=callbacks_p2, verbose=1,
)
model.save(OUT / "model2_pathogen.keras")
print("Modelo guardado en", OUT / "model2_pathogen.keras")

## Calibración de thresholds por clase

Para cada clase buscamos el threshold que maximiza F1 sobre validación. El servidor de inferencia lo usa para decidir qué clases están activas en cada parche.

In [ ]:
from sklearn.metrics import f1_score, precision_recall_curve, average_precision_score


def collect_predictions(generator, model):
    y_true, y_pred = [], []
    for i in range(len(generator)):
        x_batch, y_batch = generator[i]
        y_true.append(y_batch)
        y_pred.append(model.predict(x_batch, verbose=0))
    return np.concatenate(y_true), np.concatenate(y_pred)


y_true_val, y_pred_val = collect_predictions(val_gen, model)
y_true_binary = (y_true_val > 0.5).astype(int)

_MIN_POS = 10
_THR_LO, _THR_HI = 0.10, 0.50


def best_threshold_for_class(y_true_col, y_pred_col, lo=_THR_LO, hi=_THR_HI, min_pos=_MIN_POS):
    precisions, recalls, thresholds = precision_recall_curve(y_true_col, y_pred_col)
    if len(thresholds) == 0:
        return 0.5
    f1_values = 2 * precisions * recalls / np.clip(precisions + recalls, 1e-9, None)
    t = float(thresholds[int(np.argmax(f1_values[:-1]))]) if len(f1_values) > 1 else 0.5
    n_pos = int(y_true_col.sum())
    return float(np.clip(t, lo, hi)) if n_pos < min_pos else t


thresholds = {}
report = {}
for i, name in enumerate(CLASS_NAMES):
    n_pos = int(y_true_binary[:, i].sum())
    t = best_threshold_for_class(y_true_binary[:, i], y_pred_val[:, i])
    thresholds[name] = round(t, 3)
    pred_col = (y_pred_val[:, i] >= t).astype(int)
    report[name] = {
        "threshold": round(t, 3),
        "n_pos": n_pos,
        "f1": round(float(f1_score(y_true_binary[:, i], pred_col, zero_division=0)), 3),
        "ap": round(float(average_precision_score(y_true_binary[:, i], y_pred_val[:, i])), 3),
    }
    _flag = " [guarda clase rara]" if n_pos < _MIN_POS else ""
    print(f"{name:20s}  n_pos={n_pos:4d}  thr={t:.3f}  F1={report[name]['f1']:.3f}  AP={report[name]['ap']:.3f}{_flag}")

with open(OUT / "thresholds.json", "w") as f:
    json.dump(thresholds, f, indent=2)
with open(OUT / "calibration_report.json", "w") as f:
    json.dump(report, f, indent=2)

f1_macro = float(np.mean([v["f1"] for v in report.values()]))
map_macro = float(np.mean([v["ap"] for v in report.values()]))
print(f"\nF1 macro: {f1_macro:.3f}  |  mAP: {map_macro:.3f}")

In [ ]:
def diagnose_bias_variance_m2(h1, h2, model_name='M2'):
    all_loss  = h1.history['loss'] + h2.history['loss']
    all_vloss = h1.history['val_loss'] + h2.history['val_loss']
    all_auc   = h1.history['auc'] + h2.history['auc']
    all_vauc  = h1.history['val_auc'] + h2.history['val_auc']

    train_loss = all_loss[-1]
    val_loss   = all_vloss[-1]
    gap        = val_loss - train_loss
    val_auc    = all_vauc[-1]

    print(f'[{model_name}] train_loss={train_loss:.4f} | val_loss={val_loss:.4f} | gap={gap:.4f}')
    print(f'[{model_name}] val_auc={val_auc:.4f} (objetivo >= 0.90)')
    if train_loss > 1.0:
        print('  ALTO BIAS — underfitting. Mas epocas, mas capacidad, menos regularizacion.')
    elif gap > 0.15:
        print('  ALTA VARIANZA — overfitting. Mas dropout, L2, reducir CutMix/MixUp prob.')
    else:
        print('  Bias/varianza balanceados.')


_h1 = globals().get('h1', None)
_h2 = globals().get('h2', None)
if _h1 is not None and _h2 is not None:
    diagnose_bias_variance_m2(_h1, _h2, 'M2')

In [ ]:
import matplotlib.pyplot as plt

_h1 = globals().get("h1", None)
_h2 = globals().get("h2", None)
if _h1 is None or _h2 is None:
    print("Ejecuta primero las celdas de entrenamiento (FASE 1 y FASE 2)")
else:
    loss_h = _h1.history["loss"] + _h2.history["loss"]
    vloss = _h1.history["val_loss"] + _h2.history["val_loss"]
    auc = _h1.history["auc"] + _h2.history["auc"]
    vauc = _h1.history["val_auc"] + _h2.history["val_auc"]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))
    ep = range(1, len(loss_h) + 1)
    div = len(_h1.history["loss"])
    ax1.plot(ep, loss_h, "b-", label="train")
    ax1.plot(ep, vloss, "r-", label="val")
    ax1.axvline(div, color="gray", linestyle="--", label="fine-tune")
    ax1.set_xlabel("Epoca"); ax1.set_ylabel("Focal loss"); ax1.legend(); ax1.grid(alpha=0.3)
    ax2.plot(ep, auc, "b-", label="train")
    ax2.plot(ep, vauc, "r-", label="val")
    ax2.axvline(div, color="gray", linestyle="--")
    ax2.set_xlabel("Epoca"); ax2.set_ylabel("AUC multi-label"); ax2.legend(); ax2.grid(alpha=0.3)
    plt.tight_layout()
    OUT = globals().get("OUT", __import__("pathlib").Path("./outputs"))
    plt.savefig(OUT / "model2_pathogen_curves.png", dpi=120)
    plt.show()